# 特征提取

输出 `features/<model>.npz`（`image_ids`、`features`）。先运行初始化格；其余各格只依赖它。路径在 `fmspace/paths.py`。

In [ ]:
import pandas as pd

from fmspace.analysis import l2
from fmspace.data import load_metadata
from fmspace.extract import extract, load_features, save_features
from fmspace.models import MODELS, load_encoder
from fmspace.paths import DATA_DIR, FEATURE_DIR, OLD_FEATURE_DIR, RESULT_DIR

FEATURE_DIR.mkdir(exist_ok=True)
metadata = load_metadata(DATA_DIR)

## 试跑

In [6]:
X = extract(load_encoder("flair"), metadata["path"][:8])
X.shape

100%|██████████| 1/1 [00:01<00:00,  1.71s/it]


(8, 512)

## 全量提取

In [7]:
for name in MODELS:
    X = extract(load_encoder(name), metadata["path"])
    save_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"], X)

100%|██████████| 38/38 [04:03<00:00,  6.41s/it]


## 和旧特征比较

L2 归一化后逐张算余弦相似度。

In [8]:
cos = pd.DataFrame(
    {
        name: (
            l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
            * l2(load_features(OLD_FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
        ).sum(axis=1)
        for name in MODELS
    }
)
summary = cos.describe().T[["min", "50%"]]
summary.to_csv(RESULT_DIR / "feature_check.csv")
summary

,min,50%
siglip,0.999999,1.0
medsiglip,0.999999,1.0
conceptclip,0.999999,1.0
flair,0.999999,1.0
